# Fase 8 — Fluxograma do método e aplicação web

**Detecção de notícias falsas em português e robustez frente à IA generativa**  
Alice Vitória Cordeiro França — IFPR Campus Paranaguá  
Tecnologia em Análise e Desenvolvimento de Sistemas

---

### O que este notebook faz

Fecha o projeto com duas entregas. A primeira é o fluxograma do método, desenhado em código para que mude junto com o pipeline em vez de envelhecer num editor de imagens — já sem a etapa de Power BI, retirada do escopo. A segunda é a aplicação Gradio que permite à banca submeter um texto e ver a classificação dos três modelos, com os gráficos interativos em Plotly.

### Entradas

- `dados/metricas_consolidadas.csv`
- `dados/metricas_corpus.csv`

### Saídas de dados

- `aplicação Gradio com endereço público temporário`

### Figuras geradas para o artigo

- `figuras/fig_fluxograma_pipeline.png` — as oito etapas do método, desenhadas em código
- `figuras/fig_sintese.png` — painel de síntese dos resultados do trabalho

Todas as figuras saem deste notebook, em PNG de 400 dpi e 15 cm de largura, prontas para entrar no artigo em escala 1:1. O padrão visual está no módulo `estilo_sbc.py`, gravado no Drive pela célula 2 e idêntico em todas as fases.

## 1. Ambiente

In [ ]:
# Monta o Google Drive e fixa a estrutura de pastas do projeto.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE    = Path('/content/drive/MyDrive/TCC')
DADOS   = BASE / 'dados'
FIGURAS = BASE / 'figuras'
MODELOS = BASE / 'modelos'
for pasta in (DADOS, FIGURAS, MODELOS):
    pasta.mkdir(parents=True, exist_ok=True)

print('projeto :', BASE)
print('dados   :', DADOS)
print('figuras :', FIGURAS)
print('modelos :', MODELOS)


## 2. Padrão visual das figuras

Mesmo módulo das demais fases: a célula grava `estilo_sbc.py` e a seguinte o importa. O cabeçalho do módulo justifica cada decisão visual em relação ao template da SBC.

In [ ]:
%%writefile /content/estilo_sbc.py
# -*- coding: utf-8 -*-
"""
estilo_sbc - padrao visual unico para todas as figuras do artigo.

Cada decisao abaixo responde a uma exigencia concreta do template da SBC.

1. Largura de exatamente 15 cm, garantida no arquivo final. E a area util de
   texto do template em A4 com margens laterais de 3 cm. Nao basta pedir
   `figsize`: o `bbox_inches='tight'` do matplotlib recorta ou expande a
   figura conforme o tamanho dos rotulos, e o PNG sai com largura
   imprevisivel - entre 12 e 17 cm, nos testes. A funcao `salvar` normaliza:
   mede o resultado, completa com branco quando falta e reduz quando sobra,
   e grava a resolucao nos metadados do PNG. O Word e o LaTeX passam a
   inserir a imagem no tamanho fisico certo sem ajuste manual, e so por isso
   a afirmacao de escala 1:1 se sustenta.
2. Tipografia entre 9 e 10,5 pt no grafico. Em escala 1:1 isso fica
   proximo do corpo do texto (Times 12 pt): legivel, sem competir com ele.
3. 400 dpi em PNG. O minimo usual para impressao e 300 dpi.
4. Paleta azul / vermelhao / ambar, derivada de Okabe-Ito, que e a
   referencia de cores seguras para daltonismo em figuras cientificas. O par
   azul-laranja e o mais comum na literatura justamente por ser o que
   sobrevive as tres formas de daltonismo. As luminancias valem 0,27, 0,47 e
   0,71: o template preve avaliacao impressa, eventualmente em preto e
   branco, e cores de luminancia proxima colapsam no mesmo cinza.
5. Escala tipografica fechada, de T_PEQUENO a T_DESTAQUE. Toda figura usa
   esses cinco tamanhos e nenhum outro: tamanhos escolhidos caso a caso
   fazem o conjunto parecer montado por pessoas diferentes.
6. Nenhum texto dentro da area de dados. Rotulo de serie vai para a legenda
   abaixo do eixo, valor de referencia vai para a linha de apoio do titulo.
   Texto sobre os dados colide assim que os dados mudam - e o que fez a
   primeira versao destas figuras ficar ilegivel com o corpus real.
7. Moldura aberta (sem as linhas superior e direita), grade apenas no eixo
   de valores e rotulo numerico em cada elemento: o leitor le o valor em
   vez de estima-lo pela altura da barra.
8. Eixo de valores sempre a partir de zero nos graficos de barra. Quando a
   diferenca entre as series e pequena demais para aparecer, troca-se o
   grafico de barras por um dot plot - truncar o eixo exageraria a
   diferenca e seria enganoso.

Uso:
    from estilo_sbc import *
    destino('/content/drive/MyDrive/TCC/figuras')
    fig, ax = plt.subplots(figsize=(LARGURA, 7*CM))
    ...
    salvar(fig, 'fig_exemplo')
"""

import io
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, MaxNLocator
from PIL import Image

# ----------------------------------------------------------------- paleta
# Azul, vermelhao e ambar, da familia Okabe-Ito. O azul foi escurecido em
# relacao ao original para abrir distancia de luminancia dos outros dois e
# alcancar contraste de texto sobre branco.
AZUL    = '#1B4F72'   # serie primaria   (Regressao Logistica) - luminancia 0,27
LARANJA = '#D55E00'   # serie secundaria (Random Forest)       - luminancia 0,47
AMBAR   = '#E8B04B'   # serie terciaria  (BERTimbau)           - luminancia 0,71
VERDE   = '#117733'   # acento, so quando existe uma quarta serie
TEXTO   = '#1A1A1A'
SUAVE   = '#5F6B7A'
GRADE   = '#E4E7EA'
NEUTRO  = '#C3C9D0'

PALETA  = [AZUL, LARANJA, AMBAR, VERDE]

# Variantes escurecidas para TEXTO. A luminancia que separa as cores na
# impressao em cinza e justamente o que derruba o contraste delas sobre o
# branco: o vermelhao alcanca 3,9:1 e o ambar 1,9:1, abaixo do minimo de
# 4,5:1 que a WCAG pede para texto deste corpo. Preencher formas com a cor da
# serie e escrever com a variante escura resolve os dois lados.
AZUL_TEXTO    = AZUL       # 8,8:1
LARANJA_TEXTO = '#A64A00'  # 5,8:1
AMBAR_TEXTO   = '#8A6410'  # 5,1:1
VERDE_TEXTO   = '#0E5C28'

_LEGIVEL = {AZUL: AZUL_TEXTO, LARANJA: LARANJA_TEXTO,
            AMBAR: AMBAR_TEXTO, VERDE: VERDE_TEXTO}

# ------------------------------------------------------------ tipografia
# Escala fechada. Nenhuma figura inventa um tamanho fora destes cinco.
T_PEQUENO  = 8.0    # nota de rodape dentro da figura, anotacao secundaria
T_CORPO    = 9.0    # eixos, legenda, linha de apoio do titulo
T_ROTULO   = 9.5    # valor escrito sobre barra ou ponto
T_TITULO   = 10.5   # titulo da figura e de cada painel
T_DESTAQUE = 13.0   # numero isolado em painel de sintese


def legivel(cor):
    """Variante da cor com contraste suficiente para texto sobre branco."""
    return _LEGIVEL.get(cor, cor)


CORES_MODELO = {
    'Regressao Logistica': AZUL,
    'Regressão Logística': AZUL,
    'Random Forest': LARANJA,
    'BERTimbau': AMBAR,
}
CORES_CLASSE = {'Verdadeira': AZUL, 'Falsa': LARANJA}

# Comeca num cinza muito claro, nao em branco puro: com branco as celulas
# de valor zero somem e a matriz perde a forma de matriz.
MAPA_FRIO = LinearSegmentedColormap.from_list('sbc_frio', ['#F1F4F5', AZUL])
MAPA_DIV  = LinearSegmentedColormap.from_list('sbc_div', [AZUL, '#FFFFFF', LARANJA])

# ----------------------------------------------------------------- medidas
CM = 1 / 2.54
LARGURA = 15 * CM          # largura total da area de texto do template
LARGURA_MEIA = 7.2 * CM    # duas figuras lado a lado

# ----------------------------------------------------------------- tipografia
# DejaVu Sans vem instalada no Colab e cobre toda a acentuacao do portugues,
# mas tem apenas dois pesos: normal e negrito. Pedir 'semibold' faz o
# matplotlib emitir aviso e cair no negrito de qualquer forma, por isso os
# pesos aqui sao sempre 'normal' ou 'bold'. Se uma familia com mais pesos
# estiver instalada no ambiente, ela e usada - sem exigir instalacao nenhuma.
_instaladas = {f.name for f in mpl.font_manager.fontManager.ttflist}
FAMILIA = next((f for f in ('Inter', 'Source Sans 3', 'Lato', 'Open Sans',
                            'Liberation Sans', 'DejaVu Sans')
                if f in _instaladas), 'sans-serif')

# ----------------------------------------------------------------- rcParams
mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 400,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',

    'font.family': FAMILIA,
    'font.size': T_CORPO,
    'axes.titlesize': T_TITULO,
    'axes.titleweight': 'bold',
    'axes.titlepad': 12,
    'axes.labelsize': T_CORPO,
    'axes.labelcolor': TEXTO,
    'axes.labelpad': 10,
    'xtick.labelsize': T_CORPO,
    'ytick.labelsize': T_CORPO,
    'legend.fontsize': T_CORPO,

    'text.color': TEXTO,
    'axes.edgecolor': '#B9BDC2',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,

    'xtick.color': SUAVE,
    'ytick.color': SUAVE,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 0,
    'ytick.major.size': 0,
    'xtick.major.pad': 5,
    'ytick.major.pad': 5,

    'grid.color': GRADE,
    'grid.linewidth': 0.8,
    'axes.grid': False,

    'legend.frameon': False,
    'legend.handlelength': 1.2,
    'legend.handleheight': 1.2,
    'legend.columnspacing': 2.2,
    'legend.borderpad': 0.1,
    'legend.labelspacing': 0.6,

    'axes.prop_cycle': mpl.cycler(color=PALETA),
    'lines.linewidth': 2.0,
    'lines.markersize': 6,
    'patch.linewidth': 0,
})

# ----------------------------------------------------------------- destino
_destino = Path('.')


def destino(pasta):
    """Define a pasta onde salvar() grava as figuras."""
    global _destino
    _destino = Path(pasta)
    _destino.mkdir(parents=True, exist_ok=True)
    return _destino


def salvar(fig, nome, largura_cm=15.0, dpi=400):
    """Grava a figura em PNG com largura fisica exata e dpi nos metadados.

    O matplotlib com bbox_inches='tight' devolve larguras diferentes a cada
    figura, conforme o tamanho dos rotulos. Aqui o resultado e normalizado
    para `largura_cm`: se ficou estreito, completa-se com branco, o que
    preserva o corpo das letras; se ficou largo, reduz-se a imagem inteira,
    e a reducao aplicada e informada na saida. Passe largura_cm=None para
    gravar sem normalizar.
    """
    caminho = _destino / (nome + '.png')
    if largura_cm is None:
        fig.savefig(caminho, dpi=dpi, bbox_inches='tight', facecolor='white')
        print('[figura]', caminho.name, '(sem normalizacao)')
        return caminho

    alvo = int(round(largura_cm / 2.54 * dpi))
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight',
                facecolor='white')
    buf.seek(0)
    im = Image.open(buf).convert('RGB')
    nota = ''
    if im.width > alvo:
        fator = alvo / im.width
        im = im.resize((alvo, max(1, int(round(im.height * fator)))),
                       Image.LANCZOS)
        nota = '  (reduzida {:.1%})'.format(1 - fator)
    elif im.width < alvo:
        tela = Image.new('RGB', (alvo, im.height), 'white')
        tela.paste(im, ((alvo - im.width) // 2, 0))
        im = tela
        nota = '  (margem branca completada)'
    im.save(caminho, dpi=(dpi, dpi))
    print('[figura] {:<30} {:.1f} x {:.1f} cm{}'
          .format(caminho.name, largura_cm, im.height / dpi * 2.54, nota))
    return caminho


# ----------------------------------------------------------------- helpers
def br(texto):
    """Converte um numero ja formatado para a convencao brasileira.

    Troca o separador decimal por virgula e o de milhar por ponto. Um artigo
    em portugues com "97.40%" no grafico e "97,40%" no texto nao passa pela
    revisao, entao a conversao fica embutida nos helpers de rotulo e nos
    formatadores de eixo - nao depende de lembrar caso a caso.
    """
    return str(texto).replace(',', '\u0000').replace('.', ',').replace('\u0000', '.')


def pct(valor, dec=1):
    """Percentual na convencao brasileira: pct(0.974, 2) -> '97,40%'."""
    return br(('{:.' + str(dec) + 'f}').format(100 * valor)) + '%'


def mil(valor, dec=0):
    """Inteiro com separador de milhar: mil(11902) -> '11.902'."""
    return br(('{:,.' + str(dec) + 'f}').format(valor))


def grade_y(ax, atras=True):
    """Grade horizontal apenas (para barras verticais e linhas)."""
    ax.grid(axis='y', color=GRADE, linewidth=0.8)
    ax.grid(axis='x', visible=False)
    ax.set_axisbelow(atras)
    return ax


def grade_x(ax, atras=True):
    """Grade vertical apenas (para barras horizontais)."""
    ax.grid(axis='x', color=GRADE, linewidth=0.8)
    ax.grid(axis='y', visible=False)
    ax.set_axisbelow(atras)
    return ax


def sem_grade(ax):
    ax.grid(False)
    for lado in ('top', 'right', 'left', 'bottom'):
        ax.spines[lado].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    return ax


def eixo_percentual(ax, eixo='y', decimais=0):
    f = FuncFormatter(lambda v, _: pct(v, decimais))
    (ax.yaxis if eixo == 'y' else ax.xaxis).set_major_formatter(f)
    return ax


def eixo_decimal(ax, eixo='y', decimais=2):
    """Eixo numerico comum, tambem com virgula decimal."""
    f = FuncFormatter(
        lambda v, _: br(('{:,.' + str(decimais) + 'f}').format(v)))
    (ax.yaxis if eixo == 'y' else ax.xaxis).set_major_formatter(f)
    return ax


def escala_percentual(ax, maximo=1.0, folga_topo=0.20, passo=0.2, decimais=0):
    """Eixo percentual com folga no topo sem inventar marcas acima do maximo.

    Abrir espaco apenas com set_ylim faz o matplotlib desenhar marcas de 110%
    ou 120% num eixo de acuracia, o que nao existe. Aqui a folga e so espaco
    em branco: as marcas param no maximo real.
    """
    import numpy as _np
    ax.set_ylim(0, maximo * (1 + folga_topo))
    ax.set_yticks(_np.round(_np.arange(0, maximo + 1e-9, passo), 10))
    eixo_percentual(ax, 'y', decimais)
    return ax


def eixo_milhar(ax, eixo='y'):
    f = FuncFormatter(lambda v, _: mil(v))
    (ax.yaxis if eixo == 'y' else ax.xaxis).set_major_formatter(f)
    return ax


def rotular(ax, barras, fmt='{:.1%}', dy=5, tam=T_ROTULO, peso='bold', cor=None):
    """Escreve o valor acima de cada barra vertical."""
    for b in barras:
        v = b.get_height()
        ax.annotate(br(fmt.format(v)),
                    xy=(b.get_x() + b.get_width() / 2, v),
                    xytext=(0, dy), textcoords='offset points',
                    ha='center', va='bottom',
                    fontsize=tam, fontweight=peso,
                    color=cor or TEXTO)
    return ax


def rotular_h(ax, barras, fmt='{:.1%}', dx=6, tam=T_ROTULO, peso='bold', cor=None):
    """Escreve o valor a direita de cada barra horizontal."""
    for b in barras:
        v = b.get_width()
        ax.annotate(br(fmt.format(v)),
                    xy=(v, b.get_y() + b.get_height() / 2),
                    xytext=(dx, 0), textcoords='offset points',
                    ha='left', va='center',
                    fontsize=tam, fontweight=peso,
                    color=cor or TEXTO)
    return ax


def encolher_para_caber(ax, textos, largura_dados, tamanho, minimo=6.5,
                        passo=0.2):
    """Reduz o corpo de um conjunto de textos ate o mais largo caber.

    Mede a largura real de cada texto na figura renderizada, em vez de
    confiar numa estimativa por numero de caracteres - a largura depende da
    fonte, do peso e de quais letras sao. O mesmo tamanho e aplicado a todos
    os textos do conjunto: encolher so o que transborda deixaria um rotulo
    menor que os vizinhos, o que aparece mais do que o problema original.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderizador = fig.canvas.get_renderer()
    base = ax.transData.transform((0, 0))[0]
    limite = ax.transData.transform((largura_dados, 0))[0] - base
    while tamanho > minimo:
        maior = max(t.get_window_extent(renderer=renderizador).width
                    for t in textos)
        if maior <= limite:
            break
        tamanho -= passo
        for t in textos:
            t.set_fontsize(tamanho)
    return tamanho


def legenda_abaixo(ax, ncols=2, afastamento=None, **kwargs):
    """Legenda centralizada abaixo do eixo, fora da area de dados.

    Posicao unica para todas as figuras do artigo, por dois motivos. O
    primeiro e consistencia: legenda ora dentro, ora fora, faz o conjunto
    parecer montado aos pedacos. O segundo e que legenda dentro do grafico
    so fica livre por acaso - ela nao sabe onde estao os dados, e basta o
    corpus mudar para ela cair em cima deles.
    """
    # O afastamento e calculado em pontos e so depois convertido para
    # fracao do eixo. Usar uma fracao fixa afasta a legenda de forma
    # diferente em cada figura - num grafico de barras horizontais, alto, a
    # mesma fracao joga a legenda muito mais para baixo do que num grafico
    # baixo, e em alguns casos ela sobe por cima do rotulo do eixo.
    altura_pt = (ax.get_position().height * ax.figure.get_figheight() * 72) or 1
    if afastamento is None:
        afastamento = -(44 if ax.get_xlabel() else 24) / altura_pt
    return ax.legend(loc='upper center', ncols=ncols,
                     bbox_to_anchor=(0.5, afastamento),
                     fontsize=T_CORPO, **kwargs)


def legenda_manual(ax, itens, ncols=None, **kwargs):
    """Legenda abaixo do eixo montada a partir de (rotulo, cor).

    Para graficos cujos artistas nao carregam rotulo - area empilhada,
    histograma desenhado em duas passagens, barras coloridas uma a uma.
    """
    from matplotlib.patches import Patch
    marcas = [Patch(facecolor=cor, label=rotulo) for rotulo, cor in itens]
    return legenda_abaixo(ax, ncols=ncols or len(itens), handles=marcas,
                          **kwargs)


def titulo(ax, principal, apoio=None):
    """Titulo em duas linhas: a afirmacao e, abaixo, a leitura de apoio.

    A linha de apoio e desenhada como anotacao acima do eixo, entao o
    afastamento do titulo precisa acomoda-la - com o `pad` padrao os dois
    se sobrepoem.
    """
    linhas_apoio = apoio.count('\n') + 1 if apoio else 0
    ax.set_title(principal, loc='left',
                 pad=(25 + 12 * (linhas_apoio - 1)) if apoio else 10)
    if apoio:
        ax.annotate(apoio, xy=(0, 1), xycoords='axes fraction',
                    xytext=(0, 7), textcoords='offset points',
                    ha='left', va='bottom', fontsize=T_CORPO, color=SUAVE)
    return ax


def espacar(valores, minimo):
    """Afasta rotulos proximos demais para que nao se sobreponham.

    Devolve posicoes verticais na mesma unidade dos valores, preservando a
    ordem original. Usada quando varias series terminam em valores vizinhos:
    sem isso os rotulos se empilham e nenhum fica legivel.
    """
    import numpy as _np
    valores = _np.asarray(valores, dtype=float)
    pos = valores.copy()
    ordem = _np.argsort(valores)
    for k in range(1, len(ordem)):
        anterior, atual = ordem[k - 1], ordem[k]
        if pos[atual] - pos[anterior] < minimo:
            pos[atual] = pos[anterior] + minimo
    deslocamento = pos[ordem].mean() - valores[ordem].mean()
    return pos - deslocamento


def folga(ax, eixo='y', fator=0.14):
    """Abre espaco no topo para os rotulos nao encostarem na borda."""
    if eixo == 'y':
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + (hi - lo) * fator)
    else:
        lo, hi = ax.get_xlim()
        ax.set_xlim(lo, hi + (hi - lo) * fator)
    return ax


def painel_confusao(ax, matriz, titulo_painel, rotulos=('Verdadeira', 'Falsa'),
                    destaque=None):
    """Matriz de confusao 2x2 com contagem e percentual por celula."""
    import numpy as np
    matriz = np.asarray(matriz, dtype=float)
    total = matriz.sum()
    ax.imshow(matriz / matriz.max(), cmap=MAPA_FRIO, vmin=0, vmax=1.15)
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            v = matriz[i, j]
            claro = (v / matriz.max()) > 0.55
            ax.text(j, i - 0.10, mil(v),
                    ha='center', va='center', fontsize=T_DESTAQUE,
                    fontweight='bold',
                    color='white' if claro else TEXTO)
            ax.text(j, i + 0.22, pct(v / total, 1),
                    ha='center', va='center', fontsize=T_PEQUENO,
                    color='#E8EEF0' if claro else SUAVE)
    ax.set_xticks([0, 1], ['Predita\n' + rotulos[0], 'Predita\n' + rotulos[1]])
    ax.set_yticks([0, 1], ['Real\n' + rotulos[0], 'Real\n' + rotulos[1]])
    ax.set_title(titulo_painel, loc='center', pad=10,
                 color=legivel(destaque) if destaque else TEXTO)
    for lado in ('top', 'right', 'left', 'bottom'):
        ax.spines[lado].set_visible(False)
    ax.tick_params(length=0)
    ax.grid(False)
    return ax


def barras_pareadas(ax, categorias, series, fmt='{:.1%}', largura_grupo=0.78,
                    rotulo=True):
    """Barras agrupadas. `series` e uma lista de (nome, valores, cor)."""
    import numpy as np
    x = np.arange(len(categorias))
    n = len(series)
    larg = largura_grupo / n
    for k, (nome, valores, cor) in enumerate(series):
        desloc = (k - (n - 1) / 2) * larg
        b = ax.bar(x + desloc, valores, larg * 0.92, label=nome, color=cor)
        if rotulo:
            rotular(ax, b, fmt=fmt, tam=T_PEQUENO)
    ax.set_xticks(x, categorias)
    grade_y(ax)
    return ax


print('estilo_sbc carregado  |  paleta', AZUL, LARANJA, AMBAR)


In [ ]:
# Copia o modulo para o Drive (fica versionado junto com o projeto) e importa.
import shutil, sys
shutil.copy('/content/estilo_sbc.py', BASE / 'estilo_sbc.py')
sys.path.insert(0, '/content')

from estilo_sbc import *      # noqa: F401,F403
destino(FIGURAS)

import numpy as np
import pandas as pd


## 3. Localizadores de arquivos e colunas

In [ ]:
# Os arquivos podem estar em TCC/ ou em TCC/dados/; procura nos dois.

def arquivo(*nomes):
    """Primeiro dos nomes que existir em dados/ ou na raiz."""
    tentados = []
    for nome in nomes:
        for pasta in (DADOS, BASE):
            alvo = pasta / nome
            tentados.append(str(alvo))
            if alvo.exists():
                return alvo
    raise FileNotFoundError(
        'Nenhum destes arquivos foi encontrado:\n  ' + '\n  '.join(tentados))


def existe(*nomes):
    try:
        return arquivo(*nomes)
    except FileNotFoundError:
        return None


def coluna(df, *candidatos, obrigatoria=True):
    """Resolve o nome real de uma coluna a partir de varios candidatos."""
    mapa = {c.strip().lower(): c for c in df.columns}
    for cand in candidatos:
        if cand.strip().lower() in mapa:
            return mapa[cand.strip().lower()]
    for cand in candidatos:
        chave = cand.strip().lower()
        for norm, orig in mapa.items():
            if chave in norm:
                return orig
    if obrigatoria:
        raise KeyError('Nenhuma destas colunas existe: ' + ', '.join(candidatos)
                       + '\nDisponiveis: ' + ', '.join(map(str, df.columns)))
    return None


def csvs_do_projeto():
    """Todos os CSV sob a pasta do projeto, com contagem de linhas e colunas."""
    achados = []
    for caminho in sorted(BASE.rglob('*.csv')):
        try:
            cabecalho = list(pd.read_csv(caminho, nrows=0).columns)
            with open(caminho, encoding='utf-8', errors='ignore') as fh:
                linhas = max(sum(1 for _ in fh) - 1, 0)
        except Exception as erro:
            cabecalho, linhas = ['<ilegível: {}>'.format(erro)], -1
        achados.append((caminho, linhas, cabecalho))
    return achados


def listar_csvs():
    """Inventário dos CSV da pasta do projeto."""
    achados = csvs_do_projeto()
    if not achados:
        print('nenhum CSV encontrado em', BASE)
        return
    for caminho, linhas, colunas in achados:
        print('{:<44} {:>7} linhas   {}'.format(
            str(caminho.relative_to(BASE)), linhas,
            ', '.join(map(str, colunas))[:88]))


def procurar_csv(*pistas_de_coluna, nomes=()):
    """Acha um CSV pelo nome e, falhando isso, pelas colunas."""
    if nomes:
        alvo = existe(*nomes)
        if alvo is not None:
            return alvo
    candidatos = []
    for caminho, linhas, colunas in csvs_do_projeto():
        normalizadas = {str(c).strip().lower() for c in colunas}
        if any(any(pista in c for c in normalizadas) for pista in pistas_de_coluna):
            candidatos.append((linhas, caminho))
    if candidatos:
        candidatos.sort(reverse=True)
        escolhido = candidatos[0][1]
        print('localizado pelo conteúdo:', escolhido.relative_to(BASE))
        if len(candidatos) > 1:
            print('outros compatíveis:',
                  [str(c.relative_to(BASE)) for _, c in candidatos[1:]])
        return escolhido
    print('Nenhum CSV com coluna parecida com', pistas_de_coluna, 'foi encontrado.')
    print('Nomes procurados:', nomes or '(nenhum)')
    print('\nCSVs disponíveis em', BASE, ':')
    listar_csvs()
    raise FileNotFoundError(
        'Veja o inventário acima e renomeie o arquivo certo, ou gere-o de novo.')


def modelos_disponiveis():
    """(pasta, data, convencao) de cada modelo salvo sob o projeto.

    A convencao vem de convencao_rotulo.json, gravado pela Fase 5: sem ela
    nao da para saber se o rotulo 1 do modelo significa falsa ou verdadeira.
    """
    import datetime as _dt
    import json as _json
    achados = []
    for config in sorted(BASE.rglob('config.json')):
        pasta = config.parent
        if not (pasta / 'model.safetensors').exists() \
                and not (pasta / 'pytorch_model.bin').exists():
            continue
        marca = pasta / 'convencao_rotulo.json'
        convencao = None
        if marca.exists():
            try:
                convencao = _json.loads(marca.read_text(encoding='utf-8'))
            except Exception:
                convencao = {'erro': 'json ilegivel'}
        achados.append((pasta,
                        _dt.datetime.fromtimestamp(config.stat().st_mtime),
                        convencao))
    return sorted(achados, key=lambda t: t[1], reverse=True)


def listar_modelos():
    achados = modelos_disponiveis()
    if not achados:
        print('nenhum modelo encontrado em', BASE)
        return
    for pasta, quando, convencao in achados:
        print('{:<46} {}  {}'.format(
            str(pasta.relative_to(BASE)), quando.strftime('%d/%m %H:%M'),
            'rótulo 1 = ' + convencao.get('1', '?') if convencao
            else 'SEM convenção declarada'))


def achar_modelo(convencao_exigida='Falsa'):
    """Pasta do BERTimbau ajustado que declare a convencao de rotulo pedida.

    Escolher por uma lista fixa de nomes nao distingue o modelo desta
    execucao de um modelo antigo salvo no mesmo caminho, e um modelo
    treinado sob a convencao oposta acerta menos que o acaso.
    """
    disponiveis = [t for t in modelos_disponiveis()
                   if 'checkpoint' not in t[0].name]
    if not disponiveis:
        return None

    corretos = [t for t in disponiveis
                if t[2] and t[2].get('1') == convencao_exigida]
    if corretos:
        return str(corretos[0][0])

    print('AVISO: nenhum modelo declara a convenção "rótulo 1 =',
          convencao_exigida, '". Usando o mais recente; confira a acurácia.')
    listar_modelos()
    return str(disponiveis[0][0])


_NLP = None


def motor_lema():
    """Carrega o spaCy português uma vez por sessão."""
    global _NLP
    if _NLP is None:
        import spacy
        try:
            _NLP = spacy.load('pt_core_news_sm', disable=['parser', 'ner'])
        except OSError:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'spacy', 'download',
                            'pt_core_news_sm'], check=True)
            _NLP = spacy.load('pt_core_news_sm', disable=['parser', 'ner'])
    return _NLP


def lematizar(textos, lote=32):
    """Reduz cada texto à sequência de lemas, como o corpus foi distribuído."""
    nlp = motor_lema()
    return [' '.join(t.lemma_ for t in doc)
            for doc in nlp.pipe(list(textos), batch_size=lote)]


def taxa_de_lema(textos, limite=400):
    """Fração de tokens alfabéticos que já coincidem com o próprio lema.

    Mede o quanto um conjunto já está na forma lematizada: o corpus fica
    acima de 0,90 e o português corrente, perto de 0,60.
    """
    nlp = motor_lema()
    iguais = total = 0
    for doc in nlp.pipe(list(textos)[:limite], batch_size=32):
        for t in doc:
            if t.is_alpha and len(t.text) > 2:
                total += 1
                iguais += t.text.lower() == t.lemma_.lower()
    return iguais / total if total else float('nan')


## 4. Figura — fluxograma do método

Desenhado com `matplotlib` em vez de uma ferramenta de diagramas por dois motivos: entra no artigo com a mesma tipografia e a mesma paleta das outras figuras, e acompanha mudanças do pipeline sem retrabalho manual. As duas colunas separam a construção dos modelos, à esquerda, da avaliação adversária, à direita.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

etapas = [
    ('1', 'Corpus FakeRecogna',       '11.902 notícias verificadas'),
    ('2', 'Preparação textual',       'subtítulo descartado'),
    ('3', 'Divisão estratificada',    '80/20, semente 42'),
    ('4', 'TF-IDF e clássicos',       'reg. logística, random forest'),
    ('5', 'Ajuste fino do BERTimbau', '3 épocas, lr 2e-5'),
    ('6', 'Notícias sintéticas',      'LLM; duas formas de texto'),
    ('7', 'Robustez e explicação',    'LIME e truncamento'),
    ('8', 'Aplicação e publicação',   'Gradio, GitHub, Hugging Face'),
]
CORES_COLUNA = [AZUL, LARANJA]

fig, ax = plt.subplots(figsize=(LARGURA, 10.6 * CM))
ax.set_xlim(0, 10); ax.set_ylim(0.55, 10.55); sem_grade(ax)

# A coluna da esquerda vai de 0,15 a 4,30 e a da direita de 5,70 a 9,85: o
# corredor central de 1,4 unidade existe para a seta que troca de coluna
# passar sem cruzar caixa nenhuma.
L, A = 4.15, 1.70
col_x = [0.15, 5.70]
linha_y = [7.95, 5.60, 3.25, 0.90]
centros = {}
titulos, apoios = [], []

for i, (num, nome, apoio) in enumerate(etapas):
    c = i // 4
    x, y = col_x[c], linha_y[i % 4]
    cor = CORES_COLUNA[c]
    ax.add_patch(FancyBboxPatch((x, y), L, A,
                                boxstyle='round,pad=0.015,rounding_size=0.16',
                                facecolor='white', edgecolor=cor, linewidth=1.5))
    ax.add_patch(FancyBboxPatch((x, y), 0.16, A,
                                boxstyle='round,pad=0,rounding_size=0.05',
                                facecolor=cor, edgecolor='none'))
    ax.text(x + 0.44, y + A * 0.62, num, fontsize=T_DESTAQUE, fontweight='bold',
            color=legivel(cor), ha='center', va='center')
    titulos.append(
        ax.text(x + 0.80, y + A * 0.66, nome, fontsize=T_CORPO,
                fontweight='bold', color=TEXTO, ha='left', va='center'))
    apoios.append(
        ax.text(x + 0.80, y + A * 0.26, apoio, fontsize=T_PEQUENO, color=SUAVE,
                ha='left', va='center'))
    centros[i] = (x, y, L, A)


# Medir o texto renderizado e a unica forma confiavel de saber se ele cabe:
# a largura depende da fonte e de quais letras sao, nao do numero de
# caracteres.
UTIL = L - 0.80 - 0.18
encolher_para_caber(ax, titulos, UTIL, T_CORPO)
encolher_para_caber(ax, apoios, UTIL, T_PEQUENO)


def flecha(a, b, cor, largura=1.3):
    xa, ya, La, Aa = centros[a]
    xb, yb, Lb, Ab = centros[b]
    if abs(xa - xb) < 0.1:                       # mesma coluna: desce
        p0, p1 = (xa + La / 2, ya), (xb + Lb / 2, yb + Ab)
    else:                                        # troca de coluna: sobe pelo meio
        p0, p1 = (xa + La, ya + Aa / 2), (xb, yb + Ab / 2)
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=12,
                                 color=cor, linewidth=largura,
                                 connectionstyle='arc3,rad=0',
                                 shrinkA=3, shrinkB=3, zorder=1))


for i in (0, 1, 2):
    flecha(i, i + 1, AZUL)
flecha(3, 4, '#9AA0A6', largura=1.1)             # transição entre colunas
for i in (4, 5, 6):
    flecha(i, i + 1, LARANJA)

ax.text(col_x[0], 10.15, 'CONSTRUÇÃO DOS MODELOS', fontsize=T_PEQUENO,
        fontweight='bold', color=AZUL, ha='left', va='bottom')
ax.text(col_x[1], 10.15, 'AVALIAÇÃO ADVERSÁRIA E ENTREGA', fontsize=T_PEQUENO,
        fontweight='bold', color=LARANJA_TEXTO, ha='left', va='bottom')

salvar(fig, 'fig_fluxograma_pipeline')
plt.show()


## 5. Figura — painel de síntese

Um painel único com o que o trabalho mediu: escala do corpus, desempenho no conjunto original e queda sob texto sintético. Serve à apresentação, onde o slide precisa carregar o argumento inteiro de uma vez, mais do que ao artigo, onde cada resultado tem a sua figura.

In [ ]:
c = pd.read_csv(arquivo('metricas_consolidadas.csv'), index_col=0)
corpus = pd.read_csv(arquivo('metricas_corpus.csv'), index_col=0)

fig = plt.figure(figsize=(LARGURA, 10.2 * CM))
malha = fig.add_gridspec(2, 2, height_ratios=[0.40, 1], hspace=0.52, wspace=0.34)

ax0 = fig.add_subplot(malha[0, :]); sem_grade(ax0)
total = int(corpus.loc['Total'].iloc[0]) if 'Total' in corpus.index else 11902
cartoes = [
    (mil(total), 'notícias no corpus', AZUL),
    (pct(c['acuracia'].max(), 2), 'melhor acurácia', AZUL),
    (pct(c['deteccao_lematizada'].min()), 'pior detecção', LARANJA_TEXTO),
    (mil(c['custo_descasamento_pp'].abs().max()) + ' p.p.',
     'maior custo de forma', LARANJA_TEXTO),
]
for k, (valor, legenda, cor) in enumerate(cartoes):
    x = 0.02 + k * 0.25
    ax0.text(x, 0.76, valor, fontsize=T_DESTAQUE, fontweight='bold', color=cor,
             ha='left', va='center', transform=ax0.transAxes)
    ax0.text(x, 0.24, legenda, fontsize=T_PEQUENO, color=SUAVE,
             ha='left', va='center', transform=ax0.transAxes)

ax1 = fig.add_subplot(malha[1, 0])
b = ax1.bar(c.index, c['acuracia'], color=[CORES_MODELO[i] for i in c.index],
            width=0.5)
rotular(ax1, b, fmt='{:.1%}', tam=T_PEQUENO)
escala_percentual(ax1, folga_topo=0.20); grade_y(ax1)
ax1.set_xticks(range(len(c)), ['RL', 'RF', 'BERT'])
titulo(ax1, 'Corpus original')

ax2 = fig.add_subplot(malha[1, 1])
largura = 0.38
x = np.arange(len(c))
b1 = ax2.bar(x - largura / 2, c['deteccao_natural'], width=largura,
             color=AMBAR, label='natural')
b2 = ax2.bar(x + largura / 2, c['deteccao_lematizada'], width=largura,
             color=LARANJA, label='lematizado')
rotular(ax2, b1, fmt='{:.0%}', tam=T_PEQUENO - 0.5)
rotular(ax2, b2, fmt='{:.0%}', tam=T_PEQUENO - 0.5)
escala_percentual(ax2, folga_topo=0.24); grade_y(ax2)
ax2.set_xticks(x, ['RL', 'RF', 'BERT'])
titulo(ax2, 'Sintéticas: natural | lematizado')
# Legenda dentro do painel: num gridspec, legenda abaixo do eixo estica o
# bbox salvo e a figura inteira sai deformada.
ax2.legend(loc='upper left', fontsize=T_PEQUENO - 1, frameon=False,
           handlelength=1.0, handleheight=0.9, borderpad=0.1,
           labelspacing=0.25)

fig.suptitle('A forma do texto decide mais que a arquitetura do modelo',
             x=0.02, y=1.015, ha='left', fontsize=T_TITULO, fontweight='bold')
salvar(fig, 'fig_sintese')
plt.show()


## 6. Aplicação web

Gradio com tema próprio, na mesma paleta das figuras, e Plotly para os gráficos. Roda no próprio notebook, não exige instalação na máquina de quem avalia e expõe os três modelos sobre o mesmo texto.

A entrada é lematizada antes de chegar aos modelos. Sem esse passo a aplicação devolveria "verdadeira" com alta confiança para qualquer notícia falsa, porque os modelos foram treinados sobre texto lematizado e o usuário digita português corrente — é o descasamento medido na Fase 7, e aqui ele é corrigido na porta de entrada.

In [ ]:
!pip install -q gradio plotly
import gradio as gr
import plotly.graph_objects as go
import joblib, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

c = pd.read_csv(arquivo('metricas_consolidadas.csv'), index_col=0)
vec = joblib.load(MODELOS / 'tfidf.joblib')
lr  = joblib.load(MODELOS / 'regressao_logistica.joblib')
rf  = joblib.load(MODELOS / 'random_forest.joblib')

caminho = achar_modelo()
tok = AutoTokenizer.from_pretrained(caminho)
bert = AutoModelForSequenceClassification.from_pretrained(caminho).eval()
disp = 'cuda' if torch.cuda.is_available() else 'cpu'
bert.to(disp)

PALETA_APP = {'Regressão Logística': AZUL, 'Random Forest': LARANJA,
              'BERTimbau': AMBAR}

@torch.no_grad()
def prob_bert(texto):
    ent = tok([texto], truncation=True, max_length=256,
              padding=True, return_tensors='pt').to(disp)
    return torch.softmax(bert(**ent).logits, -1)[0].cpu().numpy()

def classificar(texto):
    if not texto or len(texto.split()) < 5:
        return ('<p style="color:#6B7280">Cole um texto com pelo menos '
                'cinco palavras.</p>', None)

    # Os modelos foram treinados sobre o corpo lematizado do FakeRecogna.
    # Entregar português corrente a eles mede descasamento de forma, não
    # falsidade: a Fase 7 mostra o BERTimbau caindo de 100% para 0% de
    # detecção só por conta disso.
    entrada = lematizar([texto])[0]

    X = vec.transform([entrada])
    probs = {
        'Regressão Logística': lr.predict_proba(X)[0],
        'Random Forest': rf.predict_proba(X)[0],
        'BERTimbau': prob_bert(entrada),
    }
    cartoes = []
    for nome, p in probs.items():
        falsa = float(p[1])
        rotulo = 'FALSA' if falsa >= 0.5 else 'VERDADEIRA'
        cor = LARANJA if falsa >= 0.5 else AZUL
        cartoes.append(
            '<div style="flex:1;border:1px solid #E6E6E6;border-left:4px solid {c};'
            'border-radius:10px;padding:14px 16px;background:#fff">'
            '<div style="font-size:12px;color:#6B7280">{n}</div>'
            '<div style="font-size:20px;font-weight:700;color:{c};margin:4px 0">{r}</div>'
            '<div style="font-size:12px;color:#6B7280">confiança {p}</div>'
            '</div>'.format(c=cor, n=nome, r=rotulo,
                            p=pct(max(falsa, 1 - falsa))))
    html = ('<div style="display:flex;gap:12px;font-family:system-ui">'
            + ''.join(cartoes) + '</div>')

    fig = go.Figure()
    for nome, p in probs.items():
        fig.add_trace(go.Bar(x=[nome], y=[float(p[1])], name=nome,
                             marker_color=PALETA_APP[nome],
                             text=[pct(float(p[1]))], textposition='outside'))
    fig.add_hline(y=0.5, line_dash='dash', line_color='#6B7280')
    fig.update_layout(template='simple_white', showlegend=False,
                      yaxis=dict(title='P(falsa)', range=[0, 1.12],
                                 tickformat='.0%'),
                      height=300, margin=dict(l=10, r=10, t=30, b=10))
    return html, fig

def grafico_desempenho():
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Corpus original', x=list(c.index),
                         y=c['acuracia'], marker_color=AZUL,
                         text=[pct(v, 2) for v in c['acuracia']],
                         textposition='outside'))
    fig.add_trace(go.Bar(name='Sintéticas (forma casada)', x=list(c.index),
                         y=c['deteccao_lematizada'], marker_color=LARANJA,
                         text=[pct(v) for v in c['deteccao_lematizada']],
                         textposition='outside'))
    fig.update_layout(template='simple_white', barmode='group',
                      yaxis=dict(title='Acerto', range=[0, 1.14], tickformat='.0%'),
                      legend=dict(orientation='h', y=1.14),
                      height=420, margin=dict(l=10, r=10, t=50, b=10))
    return fig

EXEMPLOS = [
    ['O Ministério da Saúde anunciou nesta quinta-feira a ampliação da '
     'campanha de vacinação para todos os municípios do estado.'],
    ['URGENTE: descoberta substância caseira que cura a doença em três dias; '
     'médicos escondem a informação por pressão da indústria.'],
]

with gr.Blocks(theme=gr.themes.Soft(primary_hue='teal'),
               title='Detecção de notícias falsas em português') as app:
    gr.Markdown('# Detecção de notícias falsas em português\n'
                'Três modelos sobre o mesmo texto: TF-IDF com regressão '
                'logística, TF-IDF com random forest e BERTimbau ajustado. '
                'Alice Vitória Cordeiro França — IFPR Paranaguá.')
    with gr.Tab('Classificador'):
        entrada = gr.Textbox(lines=8, label='Texto da notícia',
                             placeholder='Cole aqui o título e o corpo da notícia…')
        gr.Examples(EXEMPLOS, inputs=entrada, label='Exemplos')
        botao = gr.Button('Classificar', variant='primary')
        saida_html = gr.HTML()
        saida_fig = gr.Plot(label='Probabilidade de ser falsa')
        botao.click(classificar, entrada, [saida_html, saida_fig])
    with gr.Tab('Desempenho e robustez'):
        gr.Plot(grafico_desempenho, label='Desempenho nos dois conjuntos')
        gr.Markdown(
            '**O que o gráfico mostra.** A segunda barra é a detecção sobre '
            'notícias falsas geradas por um modelo de linguagem, medida com '
            'o texto na mesma forma do corpus de treino: BERTimbau {}, '
            'regressão logística {}.\n\n'
            'Sem esse cuidado os números seriam outros. Entregue em '
            'português corrente, o mesmo conjunto leva o BERTimbau a {} de '
            'detecção — por isso esta aplicação lematiza o que você digita '
            'antes de consultar os modelos.'
            .format(pct(c.loc['BERTimbau', 'deteccao_lematizada']),
                    pct(c.loc['Regressão Logística', 'deteccao_lematizada']),
                    pct(c.loc['BERTimbau', 'deteccao_natural'])))
    with gr.Tab('Método'):
        gr.Markdown(
            '**Corpus.** FakeRecogna, 11.902 notícias em português verificadas '
            'por agências de checagem, equilibrado entre as classes.\n\n'
            '**Divisão.** 80/20 estratificada, semente 42, idêntica para os '
            'três modelos.\n\n'
            '**Limitações.** O conjunto sintético foi gerado a partir de '
            'manchetes reais e é mais longo que as falsas do corpus; o teste de '
            'truncamento da Fase 7 mede o quanto disso é efeito de '
            'comprimento. O estudo observa notícias verificadas pelas agências '
            'presentes na base, no período coberto por ela.\n\n'
            '**Uso responsável.** A ferramenta é um apoio à triagem, não um '
            'veredito sobre a veracidade de uma notícia.')

app.launch(share=True, debug=False)


## 7. Inventário das figuras

Confere o que foi efetivamente gravado na pasta `figuras/`. Rode esta célula depois de executar as oito fases: a lista é a conferência final antes de inserir as imagens no artigo.

In [ ]:
from PIL import Image

linhas = []
for p in sorted(FIGURAS.glob('fig_*.png')):
    with Image.open(p) as im:
        larg_cm = im.width / 400 * 2.54
        linhas.append({'figura': p.name,
                       'pixels': '{}x{}'.format(im.width, im.height),
                       'largura_cm': round(larg_cm, 1),
                       'kb': round(p.stat().st_size / 1024)})
inv = pd.DataFrame(linhas)
print(inv.to_string(index=False) if len(inv) else 'nenhuma figura encontrada')
print('\ntotal:', len(inv), 'figuras em', FIGURAS)

esperadas = ['fig_corpus_balanceamento', 'fig_corpus_completude',
             'fig_corpus_tamanho', 'fig_corpus_temporal', 'fig_corpus_termos',
             'fig_desempenho_classicos', 'fig_confusao_classicos',
             'fig_coeficientes_lr', 'fig_agencias', 'fig_curva_treinamento',
             'fig_confusao_bertimbau', 'fig_sinteticas_tamanho',
             'fig_lematizacao', 'fig_desempenho_geral', 'fig_descasamento',
             'fig_lime_termos', 'fig_truncamento', 'fig_fluxograma_pipeline',
             'fig_sintese']
faltando = [e for e in esperadas if not (FIGURAS / (e + '.png')).exists()]
print('faltando:', faltando if faltando else 'nenhuma')


## 8. Resumo numérico para o artigo

Imprime, de uma vez, todos os números que a parte escrita precisa: as tabelas de métricas, as matrizes de confusão recalculadas a partir das predições, a completude por campo que sustenta o descarte do subtítulo, a janela temporal e os termos extremos. Rode depois das oito fases — e interrompa a célula do Gradio antes, porque ela fica em execução enquanto a aplicação está no ar.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def secao(titulo_secao, conteudo):
    print('\n' + '=' * 74)
    print(titulo_secao)
    print('=' * 74)
    print(conteudo)

# Tabelas de métricas, na íntegra.
for nome in ['metricas_corpus.csv', 'metricas_classicos.csv',
             'metricas_bertimbau.csv', 'metricas_consolidadas.csv',
             'metricas_sinteticas.csv', 'historico_bertimbau.csv',
             'agencias.csv', 'deteccao_sinteticas.csv', 'truncamento.csv',
             'taxas_lematizacao.csv', 'vazamento_titulo.csv']:
    caminho = existe(nome)
    if caminho is None:
        print('\n[ausente]', nome, '— a fase que o gera não foi executada')
        continue
    secao(nome, pd.read_csv(caminho, index_col=0).round(4).to_string())

# Termos extremos de cada análise léxica.
for nome, peso in [('termos_discriminantes.csv', 'log_razao'),
                   ('lime_termos.csv', 'peso_medio')]:
    caminho = existe(nome)
    if caminho is None:
        continue
    t = pd.read_csv(caminho).sort_values(peso)
    k = min(10, max(len(t) // 2, 1))
    extremos = pd.concat([t.head(k), t.tail(k)]).drop_duplicates(subset='termo')
    secao(nome, extremos.round(4).to_string(index=False))

# Matrizes de confusão, recalculadas a partir das predições gravadas.
for nome in ['predicoes_classicos.csv', 'predicoes_bertimbau.csv']:
    caminho = existe(nome)
    if caminho is None:
        continue
    pr = pd.read_csv(caminho)
    y = pr['y_verdadeiro'].to_numpy()
    for col in [c for c in pr.columns if c != 'y_verdadeiro']:
        m = confusion_matrix(y, pr[col].to_numpy(), labels=[0, 1])
        secao('matriz de confusão — ' + col,
              'linhas = real (verdadeira, falsa)   colunas = predito\n'
              + str(m) + '\n\n'
              + classification_report(y, pr[col].to_numpy(),
                                      target_names=['Verdadeira', 'Falsa'],
                                      digits=4))

# Completude por campo: é o que sustenta a decisão de descartar o subtítulo.
base = pd.read_csv(arquivo('corpus_base.csv'))
bruto = pd.read_csv(arquivo('fakerecogna_bruto.csv', 'FakeRecogna.csv'))
if 'indice_origem' in base.columns:
    bruto = bruto.loc[base['indice_origem'].to_numpy()].reset_index(drop=True)

def preenchido(serie):
    texto = serie.astype(str).str.strip()
    return 1 - ((serie.isna()) | (texto == '')
                | (texto.str.lower() == 'nan')).mean()

completude = pd.DataFrame(
    {cl: [preenchido(bruto.loc[base['classe'] == cl, c]) for c in bruto.columns]
     for cl in ['Verdadeira', 'Falsa']}, index=bruto.columns)
completude['diferenca_pp'] = 100 * (completude['Verdadeira'] - completude['Falsa'])
secao('completude por campo (% preenchido)',
      (completude * [100, 100, 1]).round(1).to_string())

# Janela temporal.
prep = existe('corpus_preparado.csv')
if prep is not None:
    datas = pd.to_datetime(pd.read_csv(prep)['data'], errors='coerce')
    secao('janela temporal',
          'de {} a {}\n{} de {} registros com data recuperada ({})'
          .format(datas.min().date(), datas.max().date(),
                  mil(datas.notna().sum()), mil(len(datas)),
                  pct(datas.notna().mean())))

secao('figuras geradas',
      '\n'.join(sorted(p.name for p in FIGURAS.glob('fig_*.png'))))
